# NeuroFinance AI — Phase 2: Exploratory Data Analysis
This notebook performs exploratory data analysis on the raw `application_train.csv` dataset, analyzing shapes, types, missing values, target imbalance, and potential data anomalies.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
print("Libraries imported successfully!")

## 1. Load Dataset

In [ ]:
dataset_path = "../data/raw/application_train.csv"
df = pd.read_csv(dataset_path)
print(f"Dataset Shape: {df.shape}")

## 2. Inspect Target Class Distribution

In [ ]:
target_counts = df["TARGET"].value_counts()
default_rate = df["TARGET"].mean()

print(f"Target distribution:\n{target_counts}")
print(f"Default rate: {default_rate:.4%}")

# Plot class distribution
plt.figure(figsize=(6, 4))
sns.countplot(x="TARGET", data=df, palette="viridis")
plt.title("Target Class Distribution (0 = Repaid, 1 = Default)")
plt.ylabel("Count")
plt.show()

## 3. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = missing / len(df)
missing_df = pd.DataFrame({"Missing Count": missing, "Percentage": missing_pct}).sort_values("Missing Count", ascending=False)

print("Top 20 columns with missing values:")
print(missing_df.head(20))

# Plot missing value percentage
plt.figure(figsize=(10, 6))
missing_df.head(20)["Percentage"].plot(kind="bar", color="salmon")
plt.title("Top 20 Columns by Missing Value Percentage")
plt.ylabel("Percentage Missing")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Anomaly Detection: DAYS_EMPLOYED

In [ ]:
print("Days Employed stats:")
print(df["DAYS_EMPLOYED"].describe())

# Anomaly counts
anom_count = (df["DAYS_EMPLOYED"] == 365243).sum()
print(f"\nNumber of anomalous days (365243): {anom_count} ({anom_count/len(df):.2%})")

# Valid versus anomalous values distribution
df_clean = df.copy()
df_clean["DAYS_EMPLOYED_ANOM"] = df_clean["DAYS_EMPLOYED"] == 365243
df_clean["DAYS_EMPLOYED"].replace(365243, np.nan, inplace=True)

plt.figure(figsize=(8, 4))
sns.histplot(-df_clean["DAYS_EMPLOYED"] / 365.25, bins=30, kde=True, color="teal")
plt.title("Distribution of Employment Duration (Years, Anomaly Removed)")
plt.xlabel("Years Employed")
plt.show()

## 5. Financial Feature Engineering Check

In [ ]:
# Let's verify the engineered features we will create in Phase 3
df_feat = df.copy()
df_feat["DAYS_EMPLOYED"].replace(365243, np.nan, inplace=True)

df_feat["AGE_YEARS"] = -df_feat["DAYS_BIRTH"] / 365.25
df_feat["EMPLOYMENT_YEARS"] = -df_feat["DAYS_EMPLOYED"] / 365.25
df_feat["CREDIT_TO_INCOME"] = df_feat["AMT_CREDIT"] / df_feat["AMT_INCOME_TOTAL"]
df_feat["ANNUITY_TO_INCOME"] = df_feat["AMT_ANNUITY"] / df_feat["AMT_INCOME_TOTAL"]
df_feat["GOODS_TO_INCOME"] = df_feat["AMT_GOODS_PRICE"] / df_feat["AMT_INCOME_TOTAL"]
df_feat["INCOME_PER_FAMILY_MEMBER"] = df_feat["AMT_INCOME_TOTAL"] / df_feat["CNT_FAM_MEMBERS"]
df_feat["EXT_SOURCE_MEAN"] = df_feat[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)
df_feat["MISSING_VALUE_COUNT"] = df_feat.isnull().sum(axis=1)

features = ["AGE_YEARS", "EMPLOYMENT_YEARS", "CREDIT_TO_INCOME", "ANNUITY_TO_INCOME", "GOODS_TO_INCOME", "INCOME_PER_FAMILY_MEMBER", "EXT_SOURCE_MEAN", "MISSING_VALUE_COUNT"]
print("Engineered Financial Features Summary:")
print(df_feat[features].describe())

# Plot correlation of financial features with TARGET
plt.figure(figsize=(10, 6))
corr_target = df_feat[features + ["TARGET"]].corr()["TARGET"].sort_values(ascending=False)
corr_target.drop("TARGET").plot(kind="bar", color="purple")
plt.title("Correlation of Engineered Features with TARGET")
plt.ylabel("Correlation Coefficient")
plt.show()